# CAN Bus Data Exploration
**Dataset:** Attack-free (normal) CAN bus traffic

This notebook loads and explores the first dataset to understand its structure before building the anomaly detection model.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from pathlib import Path

## 1. Load the Dataset

In [2]:
DATA_DIR = Path(r"D:\PROJECT\STAGEKPIT\can-anomaly-detection\CAN-Intrusion Dataset")
file_path = DATA_DIR / "Attack_free_dataset.txt"

print(f"File: {file_path.name}")
print(f"Size: {file_path.stat().st_size / (1024*1024):.1f} MB")

File: Attack_free_dataset.txt
Size: 199.4 MB


In [3]:
# Preview raw lines
with open(file_path, 'r') as f:
    for i, line in enumerate(f):
        print(repr(line.strip()))
        if i >= 9:
            break

'Timestamp:          0.000000        ID: 0316    000    DLC: 8    05 20 ea 0a 20 1a 00 7f'
'Timestamp:          0.000224        ID: 0329    000    DLC: 8    d7 a7 7f 8c 11 2f 00 10'
'Timestamp:          0.000462        ID: 0080    000    DLC: 8    00 17 ea 0a 20 1a 20 43'
'Timestamp:          0.000704        ID: 0081    000    DLC: 8    7f 84 60 00 00 00 00 53'
'Timestamp:          0.000878        ID: 0120    000    DLC: 4    00 00 00 00'
'Timestamp:          0.001115        ID: 0153    000    DLC: 8    00 80 10 ff 00 ff 40 ce'
'Timestamp:          0.001366        ID: 018f    000    DLC: 8    00 29 20 00 00 45 00 00'
'Timestamp:          0.001600        ID: 0220    000    DLC: 8    ec 03 02 04 0c 00 35 10'
'Timestamp:          0.001684        ID: 0153    100    DLC: 0'
'Timestamp:          0.001928        ID: 0153    000    DLC: 8    00 80 10 ff 00 ff 40 ce'


## 2. Parse the Data

In [4]:
def parse_can_line(line):
    """Parse a single CAN bus log line."""
    pattern = r'Timestamp:\s+([\d.]+)\s+ID:\s+([\da-fA-F]+)\s+(\d+)\s+DLC:\s+(\d+)(?:\s+([\da-fA-F ]+))?'
    match = re.match(pattern, line.strip())
    if not match:
        return None
    timestamp, can_id, _, dlc, data = match.groups()
    data_bytes = data.strip().split() if data else []
    return {
        'Timestamp': float(timestamp),
        'CAN_ID': can_id,
        'DLC': int(dlc),
        'Data': data_bytes
    }

In [5]:
# Parse all lines
records = []
with open(file_path, 'r') as f:
    for line in f:
        parsed = parse_can_line(line)
        if parsed:
            records.append(parsed)

print(f"Total messages parsed: {len(records):,}")

Total messages parsed: 2,369,398


In [6]:
# Build DataFrame
df = pd.DataFrame(records)

# Expand data bytes into separate columns
max_bytes = max(len(d) for d in df['Data'])
for i in range(max_bytes):
    df[f'DATA[{i}]'] = df['Data'].apply(lambda x: int(x[i], 16) if i < len(x) else np.nan)

df.drop(columns=['Data'], inplace=True)
df.head(10)

,Timestamp,CAN_ID,DLC,DATA[0],DATA[1],DATA[2],DATA[3],DATA[4],DATA[5],DATA[6],DATA[7]
0,0.000000,0316,8,5.0,32.0,234.0,10.0,32.0,26.0,0.0,127.0
1,0.000224,0329,8,215.0,167.0,127.0,140.0,17.0,47.0,0.0,16.0
2,0.000462,0080,8,0.0,23.0,234.0,10.0,32.0,26.0,32.0,67.0
3,0.000704,0081,8,127.0,132.0,96.0,0.0,0.0,0.0,0.0,83.0
4,0.000878,0120,4,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
5,0.001115,0153,8,0.0,128.0,16.0,255.0,0.0,255.0,64.0,206.0
6,0.001366,018f,8,0.0,41.0,32.0,0.0,0.0,69.0,0.0,0.0
7,0.001600,0220,8,236.0,3.0,2.0,4.0,12.0,0.0,53.0,16.0
8,0.001684,0153,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,0.001928,0153,8,0.0,128.0,16.0,255.0,0.0,255.0,64.0,206.0


In [7]:
df.shape

(2369398, 11)

In [8]:
df.dtypes

Timestamp    float64
CAN_ID        object
DLC            int64
DATA[0]      float64
DATA[1]      float64
DATA[2]      float64
DATA[3]      float64
DATA[4]      float64
DATA[5]      float64
DATA[6]      float64
DATA[7]      float64
dtype: object

## 3. Basic Statistics

In [12]:
print(f"Unique CAN IDs: {df['CAN_ID'].nunique()}")
print(f"DLC values: {df['DLC'].unique()}")
print(f"Time range: {df['Timestamp'].min():.6f} - {df['Timestamp'].max():.6f} seconds")
print(f"Duration: {df['Timestamp'].max() - df['Timestamp'].min():.2f} seconds")

Unique CAN IDs: 45
DLC values: [8 4 0 5 2 3]
Time range: 0.000000 - 1037.590316 seconds
Duration: 1037.59 seconds


## 4. CAN ID Distribution

In [ ]:
id_counts = df['CAN_ID'].value_counts()

plt.figure(figsize=(14, 5))
id_counts.plot(kind='bar')
plt.title('Message Count per CAN ID')
plt.xlabel('CAN ID')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
print("Top 10 most frequent IDs:")
print(id_counts.head(10))

## 5. Message Timing Analysis

In [ ]:
# Time between consecutive messages
df['Time_Diff'] = df['Timestamp'].diff()

plt.figure(figsize=(12, 4))
plt.hist(df['Time_Diff'].dropna(), bins=100, edgecolor='black')
plt.title('Distribution of Time Between Consecutive Messages')
plt.xlabel('Time (seconds)')
plt.ylabel('Frequency')
plt.yscale('log')
plt.tight_layout()
plt.show()

In [ ]:
print(f"Mean inter-message time: {df['Time_Diff'].mean()*1000:.3f} ms")
print(f"Median inter-message time: {df['Time_Diff'].median()*1000:.3f} ms")
print(f"Max gap: {df['Time_Diff'].max()*1000:.3f} ms")

## 6. Data Byte Analysis

In [ ]:
data_cols = [f'DATA[{i}]' for i in range(max_bytes)]

# Value distribution for each byte position
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    if i < len(data_cols):
        df[data_cols[i]].hist(bins=50, ax=ax, edgecolor='black')
        ax.set_title(f'Byte {i} Distribution')
        ax.set_xlabel('Value')
plt.tight_layout()
plt.show()

## 7. Summary

**Key observations from this dataset:**
- Total messages, unique IDs, time span
- Which IDs are most frequent (likely critical ECU messages)
- Timing patterns (regular vs irregular intervals)
- Data byte value ranges per position

In [ ]:
summary = {
    'total_messages': len(df),
    'unique_ids': df['CAN_ID'].nunique(),
    'duration_seconds': df['Timestamp'].max() - df['Timestamp'].min(),
    'mean_interval_ms': df['Time_Diff'].mean() * 1000,
    'most_frequent_id': id_counts.index[0],
    'most_frequent_count': id_counts.iloc[0]
}

for k, v in summary.items():
    print(f"{k}: {v}")